In [17]:
# ============================================================
# Loan Default Prediction – Final Feature Engineering Pipeline
# ============================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [4]:
# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------
DATA_PATH = "../data/raw/Loan_Default.csv"   # change path if needed
df = pd.read_csv(DATA_PATH)

print("Initial Shape:", df.shape)

Initial Shape: (255347, 18)


In [5]:
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [6]:
cols_to_ignore = ['LoanID']
target_col = 'Default'
label_encode_cols = ['HasMortgage','HasDependents','HasCoSigner']
one_hot_encode_cols = ['Education', 'EmploymentType', 'MaritalStatus', 'LoanPurpose']

In [7]:
df.columns

Index(['LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore',
       'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm',
       'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus',
       'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner',
       'Default'],
      dtype='object')

In [8]:
# ------------------------------------------------------------
# 2. BASIC CLEANING
# ------------------------------------------------------------

# Remove duplicates
df.drop_duplicates(inplace=True)

In [9]:

# Drop ID-like columns
df.drop(columns=['LoanID'], inplace=True, errors="ignore")

print("After cleaning:", df.shape)

After cleaning: (255347, 17)


In [10]:
# ------------------------------------------------------------
# 3. TARGET VARIABLE
# ------------------------------------------------------------
TARGET = "Default"
X = df.drop(columns=[TARGET])
y = df[TARGET]

In [11]:
# ------------------------------------------------------------
# 4. FEATURE ENGINEERING (DERIVED FEATURES)
# ------------------------------------------------------------

# Financial pressure indicators
X["loan_interest_burden"] = X["LoanAmount"] * X["InterestRate"]
X["loan_term_pressure"] = X["LoanAmount"] / (X["LoanTerm"] + 1)

In [12]:
[col for col in df.columns if col not in ['HasCoSigner','LoanPurpose','HasDependents', 
                        'HasMortgage','MaritalStatus', 'EmploymentType', 
                        'Education']]

['Age',
 'Income',
 'LoanAmount',
 'CreditScore',
 'MonthsEmployed',
 'NumCreditLines',
 'InterestRate',
 'LoanTerm',
 'DTIRatio',
 'Default']

In [13]:
# ------------------------------------------------------------
# 5. SELECT FINAL FEATURES
# ------------------------------------------------------------

numerical_features = ['Age',
 'Income',
 'LoanAmount',
 'CreditScore',
 'MonthsEmployed',
 'NumCreditLines',
 'InterestRate',
 'LoanTerm',
 'DTIRatio']

categorical_features = ['HasCoSigner','LoanPurpose','HasDependents', 
                        'HasMortgage','MaritalStatus', 'EmploymentType', 
                        'Education']

X = X[numerical_features + categorical_features]

In [14]:
X['LoanPurpose'].unique()

array(['Other', 'Auto', 'Business', 'Home', 'Education'], dtype=object)

In [15]:
# ------------------------------------------------------------
# 6. PREPROCESSING PIPELINES
# ------------------------------------------------------------

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    # ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)


In [16]:


# ------------------------------------------------------------
# 7. MODEL PIPELINE
# ------------------------------------------------------------
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier 

lr = LogisticRegression(max_iter=1000)
xgb = XGBClassifier(eval_metric="logloss")
lgbm = LGBMClassifier()

model_pipeline = Pipeline(steps=[("preprocessor", preprocessor),
    ("model", xgb)
])


In [18]:
# ------------------------------------------------------------
# 8. TRAIN / TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,random_state=42,
    stratify=y
)


In [19]:
# ------------------------------------------------------------
# 9. TRAIN MODEL
# ------------------------------------------------------------

model_pipeline.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [20]:
# ------------------------------------------------------------
# 10. EVALUATION
# ------------------------------------------------------------

y_pred = model_pipeline.predict(X_test)
print("\nMODEL PERFORMANCE\n")
print(classification_report(y_test, y_pred))

# Predict probability for positive class (class = 1)
y_pred_lr = model_pipeline.predict_proba(X_test)[:, 1]
# Calculate AUC-ROC score
from sklearn.metrics import roc_auc_score
auc_lr = roc_auc_score(y_test, y_pred_lr)
print("Logistic Regression AUC-ROC:", auc_lr)


MODEL PERFORMANCE

              precision    recall  f1-score   support

           0       0.89      0.99      0.94     45139
           1       0.56      0.08      0.15      5931

    accuracy                           0.89     51070
   macro avg       0.72      0.54      0.54     51070
weighted avg       0.85      0.89      0.85     51070

Logistic Regression AUC-ROC: 0.7431500866640566


In [21]:
# ------------------------------------------------------------
# 11. SAVE PIPELINE
# ------------------------------------------------------------

joblib.dump(model_pipeline, "loan_default_pipeline.pkl")

print("\nModel and feature list saved successfully!")


Model and feature list saved successfully!


In [27]:
# ============================================================
# 12. INFERENCE FUNCTION (RAW USER INPUT)
# ============================================================

def predict_loan_default(raw_input: dict):
    
    model = joblib.load("loan_default_pipeline.pkl")

    user_df = pd.DataFrame([raw_input])

    # -------- Derive features for inference --------
    user_df["loan_interest_burden"] = user_df["LoanAmount"] * user_df["InterestRate"]
    user_df["loan_term_pressure"] = user_df["LoanAmount"] / (user_df["LoanTerm"] + 1)
    

    prediction = model.predict(user_df)[0]
    probability = model.predict_proba(user_df)[0][1]

    if probability < 0.3:
        risk = "Low Risk"
        action = "Send payment reminder via SMS/Email"
    elif probability < 0.6:
        risk = "Medium Risk"
        action = "Offer flexible EMI or short-term payment plan"
    else:
        risk = "High Risk"
        action = "Assign to recovery agent and initiate call"

    return {
        "default_probability": float(round(probability*100, 2)),
        "risk_level": risk,
        "recommended_action": action
    }

In [23]:
df.columns

Index(['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed',
       'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'Education',
       'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents',
       'LoanPurpose', 'HasCoSigner', 'Default'],
      dtype='object')

In [24]:
X['LoanPurpose'].unique()

array(['Other', 'Auto', 'Business', 'Home', 'Education'], dtype=object)

In [29]:
# ------------------------------------------------------------
# 13. SAMPLE INFERENCE TEST
# ------------------------------------------------------------

if __name__ == "__main__":
    sample_input = {
    "Age": 35,
    "Income": 60000,
    "LoanAmount": 250000,
    "CreditScore": 720,
    "MonthsEmployed": 60,
    "NumCreditLines": 5,
    "InterestRate": 11.5,
    "LoanTerm": 36,
    "DTIRatio": 0.35,

    "HasMortgage": "Yes",
    "HasDependents": "No",
    "HasCoSigner": "No",

    "Education": "Bachelor",
    "EmploymentType": "Salaried",
    "MaritalStatus": "Married",
    "LoanPurpose": "Home"
    }
    print("\nInference Output:")
    print(predict_loan_default(sample_input))


Inference Output:
{'default_probability': 5.570000171661377, 'risk_level': 'Low Risk', 'recommended_action': 'Send payment reminder via SMS/Email'}


In [30]:
df.columns

Index(['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed',
       'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'Education',
       'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents',
       'LoanPurpose', 'HasCoSigner', 'Default'],
      dtype='object')

In [31]:
df['LoanAmount'].unique()

array([ 50587, 124440, 129188, ..., 105905, 168231, 208294],
      shape=(158729,))

In [32]:
X['InterestRate'].unique()

array([15.23,  4.81, 21.17, ...,  2.46, 19.81,  9.01], shape=(2301,))

In [35]:
X['MaritalStatus'].unique()

array(['Divorced', 'Married', 'Single'], dtype=object)

In [26]:
X['LoanTerm'].unique()

array([36, 60, 24, 48, 12])

In [54]:
pd.DataFrame([sample_input]).columns

Index(['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed',
       'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'HasMortgage',
       'HasDependents', 'HasCoSigner', 'Education', 'EmploymentType',
       'MaritalStatus', 'LoanPurpose'],
      dtype='object')

In [59]:
np.array([35,60000,250000,720,60,5,11.5,36,0.35,"Yes","No","No",
         "Bachelor","Salaried","Married","Home"])

array(['35', '60000', '250000', '720', '60', '5', '11.5', '36', '0.35',
       'Yes', 'No', 'No', 'Bachelor', 'Salaried', 'Married', 'Home'],
      dtype='<U32')

In [60]:
pd.DataFrame([35,60000,250000,720,60,5,11.5,36,0.35,"Yes","No","No",
         "Bachelor","Salaried","Married","Home"])

,0
0,35
1,60000
2,250000
3,720
4,60
5,5
6,11.5
7,36
8,0.35
9,Yes


In [62]:
pd.DataFrame(np.array([35,60000,250000,720,60,5,11.5,36,0.35,"Yes","No","No",
         "Bachelor","Salaried","Married","Home"]).reshape(1, -1),columns=["Age","Income","LoanAmount","CreditScore","MonthsEmployed","NumCreditLines","InterestRate","LoanTerm","DTIRatio","HasMortgage","HasDependents","HasCoSigner","Education","EmploymentType","MaritalStatus","LoanPurpose"])

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,HasMortgage,HasDependents,HasCoSigner,Education,EmploymentType,MaritalStatus,LoanPurpose
0,35,60000,250000,720,60,5,11.5,36,0.35,Yes,No,No,Bachelor,Salaried,Married,Home


In [63]:
import pandas as pd
import numpy as np
import joblib
import statsmodels.api as sm

In [64]:
# Load trained pipeline
pipeline = joblib.load("loan_default_pipeline.pkl")

# Extract preprocessing step
preprocessor = pipeline.named_steps["preprocessor"]

# Load original dataset again
df = pd.read_csv("../data/raw/Loan_Default.csv")
df.head()

# Target
y = df["Default"]

# Apply SAME feature engineering as training
X = df.drop(columns=["Default"])

# Transform features
X_processed = preprocessor.transform(X)

# Add intercept
X_processed = sm.add_constant(X_processed)


In [65]:
# Numerical feature names
num_features = preprocessor.transformers_[0][2]

# Categorical feature names
cat_encoder = preprocessor.transformers_[1][1]
cat_features = cat_encoder.get_feature_names_out(
    preprocessor.transformers_[1][2]
)

# Final feature list
all_features = np.concatenate([["Intercept"], num_features, cat_features])


In [66]:
OneHotEncoder(handle_unknown="ignore", drop="first")


,categories,'auto'
,drop,'first'
,sparse_output,True
,dtype,<class 'numpy.float64'>
,handle_unknown,'ignore'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [67]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first"))
])


In [68]:
X_processed = preprocessor.transform(X)
X_processed = sm.add_constant(X_processed)

logit_model = sm.Logit(y, X_processed)
result = logit_model.fit(method="lbfgs", maxiter=200)


C:\Users\DELL\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


In [69]:
from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=1e-5)
X_processed = vt.fit_transform(X_processed)


In [70]:
import numpy as np

print("NaNs:", np.isnan(X_processed).sum())
print("Infs:", np.isinf(X_processed).sum())
print("Rank:", np.linalg.matrix_rank(X_processed))
print("Columns:", X_processed.shape[1])


NaNs: 0
Infs: 0
Rank: 25
Columns: 28


In [71]:
logit_model = sm.Logit(y, X_processed)
result = logit_model.fit(maxiter=100, disp=False)

In [72]:
print("X_processed shape:", X_processed.shape)
print("Params length:", len(result.params))


X_processed shape: (255347, 28)
Params length: 28


In [73]:
# Numerical feature names
num_features = preprocessor.transformers_[0][2]

# Categorical feature names AFTER drop="first"
cat_encoder = preprocessor.transformers_[1][1]
cat_features = cat_encoder.get_feature_names_out(
    preprocessor.transformers_[1][2]
)

# Combine feature names
feature_names = np.concatenate([num_features, cat_features])

# Add intercept manually
feature_names = np.insert(feature_names, 0, "Intercept")


In [74]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = "../data/raw/Loan_Default.csv"

cols_to_ignore = ['LoanID']
target_col = 'Default'

label_encode_cols = ['HasMortgage', 'HasDependents', 'HasCoSigner']
one_hot_encode_cols = ['Education', 'EmploymentType', 'MaritalStatus', 'LoanPurpose']

MODEL_PATH = "loan_default_pipeline.pkl"

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv(DATA_PATH)

# Drop ignored columns
df = df.drop(columns=cols_to_ignore)

X = df.drop(columns=[target_col])
y = df[target_col]

# Identify numerical columns
numerical_cols = [
    col for col in X.columns
    if col not in label_encode_cols + one_hot_encode_cols
]

# -----------------------------
# PREPROCESSING
# -----------------------------
label_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

one_hot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

preprocessor = ColumnTransformer(
    transformers=[
        ("label", label_encoder, label_encode_cols),
        ("onehot", one_hot_encoder, one_hot_encode_cols),
        ("num", StandardScaler(), numerical_cols)
    ]
)

# -----------------------------
# MODEL
# -----------------------------
model = LogisticRegression(max_iter=1000)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

# -----------------------------
# TRAIN / TEST SPLIT
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -----------------------------
# TRAIN
# -----------------------------
pipeline.fit(X_train, y_train)

# -----------------------------
# EVALUATE
# -----------------------------
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# -----------------------------
# SAVE PIPELINE
# -----------------------------
joblib.dump(pipeline, MODEL_PATH)
print(f"\nPipeline saved as {MODEL_PATH}")


Accuracy: 0.8852946935578617

Classification Report:
               precision    recall  f1-score   support

           0       0.89      1.00      0.94     45139
           1       0.61      0.03      0.06      5931

    accuracy                           0.89     51070
   macro avg       0.75      0.52      0.50     51070
weighted avg       0.85      0.89      0.84     51070


Pipeline saved as loan_default_pipeline.pkl


In [75]:
import joblib
import pandas as pd

MODEL_PATH = "loan_default_pipeline.pkl"

# Load trained pipeline
pipeline = joblib.load(MODEL_PATH)

def predict_default(raw_input: dict):
    """
    raw_input: dictionary with raw values (same as dataset columns)
    """

    # Convert dict → DataFrame
    input_df = pd.DataFrame([raw_input])

    # Predict class
    prediction = pipeline.predict(input_df)[0]

    # Predict probability (if needed)
    probability = pipeline.predict_proba(input_df)[0][1]

    return {
        "Default_Prediction": int(prediction),
        "Default_Probability": round(probability, 4)
    }


# -----------------------------
# EXAMPLE USAGE
# -----------------------------
if __name__ == "__main__":
    sample_input = {
        "Age": 35,
        "Income": 60000,
        "LoanAmount": 250000,
        "CreditScore": 720,
        "MonthsEmployed": 60,
        "NumCreditLines": 5,
        "InterestRate": 11.5,
        "LoanTerm": 36,
        "DTIRatio": 0.35,

        "HasMortgage": "Yes",
        "HasDependents": "No",
        "HasCoSigner": "No",

        "Education": "Bachelor",
        "EmploymentType": "Salaried",
        "MaritalStatus": "Married",
        "LoanPurpose": "Home"
    }

    result = predict_default(sample_input)
    print(result)


{'Default_Prediction': 0, 'Default_Probability': np.float64(0.2703)}
